# 02 — Évaluation et Comparaison des Modèles
## Projet ObRail Europe — Prévision de fréquentation & Détection des pays en déclin

**Objectif** : Produire les visualisations d'évaluation des modèles entraînés :
- Matrices de confusion (4 modèles classification)
- Courbes ROC (4 modèles classification)
- Actual vs Predicted (3 modèles régression)
- Tableaux comparatifs finaux

---
## 0. Imports et configuration

In [3]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc,
    classification_report
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

# Chemins
ROOT       = Path("../../")
DATA_ML    = ROOT / "data" / "ml"
MODELS_DIR = ROOT / "ia" / "models"
REPORTS    = ROOT / "ia" / "reports"
DOCS_DIR   = ROOT / "docs" / "ia"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

CLASSIF_DATASET_PATH    = DATA_ML / "classification_dataset.csv"
REGRESSION_DATASET_PATH = DATA_ML / "regression_dataset.csv"
PREPROCESSOR_CLF_PATH   = DATA_ML / "preprocessor_classification.joblib"
PREPROCESSOR_REG_PATH   = DATA_ML / "preprocessor_regression.joblib"

# Features (identiques à train_utils.py)
NUMERIC_FEATURES      = ["year","co2_emissions","co2_per_passenger","co2_lag1","passengers_lag1","passengers_lag2"]
CATEGORICAL_FEATURES  = ["country_name"]

print("Chemins configurés ✅")

Chemins configurés ✅


---
## 1. Chargement des données et des modèles

In [4]:
from sklearn.model_selection import train_test_split

# --- Classification ---
df_clf = pd.read_csv(CLASSIF_DATASET_PATH)
X_clf  = df_clf[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_clf  = df_clf["en_declin"]

_, X_clf_test, _, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.20, stratify=y_clf, random_state=42
)

preprocessor_clf = joblib.load(PREPROCESSOR_CLF_PATH)
X_clf_test_p     = preprocessor_clf.transform(X_clf_test)

# --- Régression ---
df_reg = pd.read_csv(REGRESSION_DATASET_PATH)
X_reg  = df_reg[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_reg  = df_reg["passengers"]

_, X_reg_test, _, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)

preprocessor_reg = joblib.load(PREPROCESSOR_REG_PATH)
X_reg_test_p     = preprocessor_reg.transform(X_reg_test)

print(f"Test clf : {X_clf_test_p.shape} | Test reg : {X_reg_test_p.shape}")

FileNotFoundError: [Errno 2] No such file or directory: '..\\..\\data\\ml\\classification_dataset.csv'

In [ ]:
# Chargement des modèles classification
clf_models = {
    "Logistic Regression" : joblib.load(MODELS_DIR / "logistic_clf.joblib"),
    "Random Forest"       : joblib.load(MODELS_DIR / "random_forest_clf.joblib"),
    "XGBoost"             : joblib.load(MODELS_DIR / "xgboost_clf.joblib"),
    "MLP"                 : joblib.load(MODELS_DIR / "mlp_clf.joblib"),
}

# Chargement des modèles régression
reg_models = {
    "Ridge"         : joblib.load(MODELS_DIR / "ridge_reg.joblib"),
    "Random Forest" : joblib.load(MODELS_DIR / "random_forest_reg.joblib"),
    "XGBoost"       : joblib.load(MODELS_DIR / "xgboost_reg.joblib"),
}

# Chargement des métriques JSON
def load_metrics(name, axis):
    path = MODELS_DIR / f"{name}_{axis}_metrics.json"
    with open(path) as f:
        return json.load(f)

clf_metrics_map = {
    "Logistic Regression" : load_metrics("logistic",      "clf"),
    "Random Forest"       : load_metrics("random_forest", "clf"),
    "XGBoost"             : load_metrics("xgboost",       "clf"),
    "MLP"                 : load_metrics("mlp",           "clf"),
}
reg_metrics_map = {
    "Ridge"         : load_metrics("ridge",         "reg"),
    "Random Forest" : load_metrics("random_forest", "reg"),
    "XGBoost"       : load_metrics("xgboost",       "reg"),
}

print("Modèles et métriques chargés ✅")

---
## 2. Matrices de confusion — Classification

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, model) in zip(axes, clf_models.items()):
    y_pred = model.predict(X_clf_test_p)
    cm     = confusion_matrix(y_clf_test, y_pred)
    disp   = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["En croissance", "En déclin"]
    )
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    f1 = clf_metrics_map[name]["f1"]
    ax.set_title(f"{name}\nF1 = {f1:.4f}", fontsize=11)
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Réel")
    ax.tick_params(axis="x", rotation=20)

plt.suptitle("Matrices de confusion — Axe Classification (en_declin)",
             fontsize=14, y=1.04)
plt.tight_layout()
plt.savefig(DOCS_DIR / "fig_confusion_matrices.png", bbox_inches="tight")
plt.show()
print("Figure sauvegardée : docs/fig_confusion_matrices.png")

---
## 3. Courbes ROC — Classification

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

colors = ["#2980b9", "#27ae60", "#e67e22", "#8e44ad"]

for (name, model), color in zip(clf_models.items(), colors):
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_clf_test_p)[:, 1]
        fpr, tpr, _ = roc_curve(y_clf_test, y_proba)
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f"{name} (AUC = {roc_auc_val:.4f})")
    else:
        ax.plot([], [], color=color, linewidth=2,
                label=f"{name} (pas de predict_proba)")

# Diagonale hasard
ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.6, label="Aléatoire (AUC = 0.50)")

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("Taux de Faux Positifs (FPR)", fontsize=12)
ax.set_ylabel("Taux de Vrais Positifs (TPR)", fontsize=12)
ax.set_title("Courbes ROC — Axe Classification (en_declin)", fontsize=14)
ax.legend(loc="lower right", fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DOCS_DIR / "fig_roc_curves.png", bbox_inches="tight")
plt.show()
print("Figure sauvegardée : docs/fig_roc_curves.png")

---
## 4. Rapport de classification détaillé

In [ ]:
for name, model in clf_models.items():
    y_pred = model.predict(X_clf_test_p)
    print(f"\n{'='*50}")
    print(f"Modèle : {name}")
    print('='*50)
    print(classification_report(
        y_clf_test, y_pred,
        target_names=["En croissance (0)", "En déclin (1)"],
        zero_division=0
    ))

---
## 5. Actual vs Predicted — Régression

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors_reg = ["#2980b9", "#27ae60", "#e67e22"]

for ax, (name, model), color in zip(axes, reg_models.items(), colors_reg):
    y_pred = model.predict(X_reg_test_p)
    m      = reg_metrics_map[name]

    ax.scatter(y_reg_test, y_pred, alpha=0.5, color=color,
               edgecolors="none", s=25)

    # Droite parfaite y = x
    max_val = max(y_reg_test.max(), y_pred.max())
    ax.plot([0, max_val], [0, max_val], "r--", linewidth=1.5, label="Prédit = Réel")

    ax.set_title(
        f"{name}\nR²={m['r2']:.4f} | MAE={m['mae']:.0f}",
        fontsize=11
    )
    ax.set_xlabel("Valeurs réelles (passengers)")
    ax.set_ylabel("Valeurs prédites")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("Actual vs Predicted — Axe Régression (passengers)",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(DOCS_DIR / "fig_actual_vs_predicted.png", bbox_inches="tight")
plt.show()
print("Figure sauvegardée : docs/fig_actual_vs_predicted.png")

---
## 6. Résidus des modèles de régression

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model), color in zip(axes, reg_models.items(), colors_reg):
    y_pred  = model.predict(X_reg_test_p)
    residus = y_reg_test.values - y_pred

    ax.scatter(y_pred, residus, alpha=0.5, color=color, edgecolors="none", s=25)
    ax.axhline(0, color="red", linestyle="--", linewidth=1.5)
    ax.set_title(f"{name} — Résidus", fontsize=11)
    ax.set_xlabel("Valeurs prédites")
    ax.set_ylabel("Résidus (réel − prédit)")
    ax.grid(True, alpha=0.3)

plt.suptitle("Analyse des résidus — Axe Régression", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(DOCS_DIR / "fig_residus_regression.png", bbox_inches="tight")
plt.show()
print("Figure sauvegardée : docs/fig_residus_regression.png")

---
## 7. Tableaux comparatifs finaux

In [ ]:
# Tableau classification
df_clf_compare = pd.DataFrame([
    {"Modèle": name, **m}
    for name, m in clf_metrics_map.items()
])
df_clf_compare = df_clf_compare.set_index("Modèle")
df_clf_compare.columns = ["Accuracy", "Precision", "Recall", "F1 ⭐", "ROC-AUC"]

print("=" * 65)
print("CLASSIFICATION — Tableau comparatif (critère principal : F1)")
print("=" * 65)
print(df_clf_compare.round(4).to_string())
print(f"\n→ Modèle sélectionné : {df_clf_compare['F1 ⭐'].idxmax()} "
      f"(F1 = {df_clf_compare['F1 ⭐'].max():.4f})")

In [ ]:
# Tableau régression
df_reg_compare = pd.DataFrame([
    {"Modèle": name, **m}
    for name, m in reg_metrics_map.items()
])
df_reg_compare = df_reg_compare.set_index("Modèle")
df_reg_compare.columns = ["MAE", "RMSE", "R² ⭐"]

print("=" * 55)
print("RÉGRESSION — Tableau comparatif (critère principal : R²)")
print("=" * 55)
print(df_reg_compare.round(4).to_string())
print(f"\n→ Modèle sélectionné : {df_reg_compare['R² ⭐'].idxmax()} "
      f"(R² = {df_reg_compare['R² ⭐'].max():.4f})")

---
## 8. Visualisation comparative des métriques

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Classification : barplot F1 ---
models_clf  = list(clf_metrics_map.keys())
metrics_clf = ["Accuracy", "Precision", "Recall", "F1"]
x = np.arange(len(models_clf))
width = 0.2
palette = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12"]

for i, (metric, color) in enumerate(zip(metrics_clf, palette)):
    values = [clf_metrics_map[m][metric.lower()] for m in models_clf]
    axes[0].bar(x + i * width, values, width, label=metric, color=color, alpha=0.85)

axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(models_clf, rotation=15, ha="right")
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Métriques Classification par modèle", fontsize=13)
axes[0].set_ylabel("Score")
axes[0].legend(fontsize=10)
axes[0].grid(True, axis="y", alpha=0.3)

# --- Régression : barplot R² ---
models_reg = list(reg_metrics_map.keys())
r2_values  = [reg_metrics_map[m]["r2"] for m in models_reg]
colors_r2  = ["#2980b9", "#27ae60", "#e67e22"]

bars = axes[1].bar(models_reg, r2_values, color=colors_r2, edgecolor="white", linewidth=1.2)
for bar, val in zip(bars, r2_values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() - 0.01,
                 f"{val:.4f}", ha="center", va="top",
                 fontsize=11, fontweight="bold", color="white")
axes[1].set_ylim(0.94, 1.001)
axes[1].set_title("R² par modèle de régression", fontsize=13)
axes[1].set_ylabel("R²")
axes[1].grid(True, axis="y", alpha=0.3)

plt.suptitle("Comparaison des modèles — Axe Classification & Régression",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(DOCS_DIR / "fig_comparaison_modeles.png", bbox_inches="tight")
plt.show()
print("Figure sauvegardée : docs/fig_comparaison_modeles.png")

---
## 9. Figures produites

| Figure | Fichier |
|--------|---------|
| Matrices de confusion (4 modèles clf) | `docs/ia/fig_confusion_matrices.png` |
| Courbes ROC (4 modèles clf) | `docs/ia/fig_roc_curves.png` |
| Actual vs Predicted (3 modèles reg) | `docs/ia/fig_actual_vs_predicted.png` |
| Analyse des résidus | `docs/ia/fig_residus_regression.png` |
| Comparaison des métriques | `docs/ia/fig_comparaison_modeles.png` |